# 🏊 Pooling — Notes + Interview
---
> **Simple English** | **Interview Ready**

## 📌 What is Pooling? (Simple English)
- Pooling = **shrinks** the feature map while **keeping the important info**
- Slides a window (usually 2×2) over the feature map and takes 1 value per window
- Makes the network **translation invariant** (small shifts don't matter)
- Reduces computation and prevents overfitting
- Two types: **Max Pooling** (most popular) and **Average Pooling**

## 🔑 Max vs Average Pooling
| | Max Pooling | Average Pooling |
|---|---|---|
| Takes | Largest value in window | Average of all values |
| Keeps | Strongest feature (edge/peak) | Smooth representation |
| Use | Most CNN layers ✅ | Final pooling (GAP) |
| Intuition | "Was the feature present?" | "How present on average?" |

## 🧱 Pooling Formula
```
Input: 4×4  →  Pool 2×2, Stride=2  →  Output: 2×2
Output = floor((Input - Pool_size) / Stride) + 1
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Manual Max Pooling and Average Pooling
def max_pool(feature_map, pool_size=2, stride=2):
    h, w = feature_map.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            window = feature_map[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i,j] = np.max(window)
    return output

def avg_pool(feature_map, pool_size=2, stride=2):
    h, w = feature_map.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            window = feature_map[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i,j] = np.mean(window)
    return output

feature_map = np.array([
    [1,  3,  2,  4],
    [5,  6,  1,  2],
    [3,  2,  7,  8],
    [1,  4,  9,  5]
], dtype=float)

mp = max_pool(feature_map)
ap = avg_pool(feature_map)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(axes,
    [feature_map, mp, ap],
    ['Feature Map (4×4)', 'Max Pool (2×2) → 2×2', 'Avg Pool (2×2) → 2×2']):
    ax.imshow(data, cmap='YlOrRd')
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, int(data[i,j]), ha='center', va='center', fontsize=14, fontweight='bold')
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()
print("Max Pool takes the LARGEST from each 2×2 window")
print("Avg Pool takes the AVERAGE from each 2×2 window")

In [ ]:
import tensorflow as tf
import numpy as np

# MaxPooling2D in Keras
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(28,28,1)),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=2),   # halves size
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='softmax')
])
model.summary()

# GlobalAveragePooling — modern alternative to Flatten
model_gap = tf.keras.Sequential([
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', input_shape=(28,28,1)),
    tf.keras.layers.GlobalAveragePooling2D(),  # 1 value per channel
    tf.keras.layers.Dense(10, activation='softmax')
])
print("\nGlobal Average Pooling output shape:", model_gap.layers[1].output_shape)
print("→ Takes average of entire feature map per channel")
print("→ No Flatten needed, fewer parameters!")

## 🗣️ Interview Q&A

**Q: What is max pooling?**
> Divides feature map into non-overlapping windows and takes the maximum value from each. Keeps the most activated (important) feature. Most common pooling type.

**Q: Why do we use pooling?**
> (1) Reduces spatial size → fewer params in next layers, (2) Creates translation invariance (feature detected slightly off-center still works), (3) Controls overfitting by reducing parameters.

**Q: What is Global Average Pooling (GAP)?**
> Takes the average of the entire feature map for each channel → produces a single value per channel. Replaces Flatten + Dense. Used in modern architectures (ResNet, MobileNet). Much fewer parameters.

**Q: Max pooling vs Average pooling — when to use which?**
> Max pooling → feature detection (is this feature present?), most hidden layers.
> Average pooling → spatial average, especially Global Average Pooling at the end of networks.